**Goal**:
- get hotel name from the xlsx file.
- search query the hotel name via the GET api/v1/hotels/searchDestinationOrHotel api
- get the entityid
- go to GET api/v1/hotels/getHotelDetails
- enter the entityid as parameter for 
    - hotelId
    - entityId
- 
check the policies.content part of the json file.
    - is it queen-or-king-bed-included
    - does it have breakfast-included
    - does it have pool-access-included
    - does it have wifi-included

```bash
curl --request GET \
	--url 'https://sky-scrapper.p.rapidapi.com/api/v1/hotels/searchDestinationOrHotel?query=RedDoorz%20Plus' \
	--header 'x-rapidapi-host: sky-scrapper.p.rapidapi.com' \
	--header 'x-rapidapi-key: f3068fa39amsha256be6eec9ae0ep135038jsnbf15340e581c'
```

Let's start by adding our pip installs

In [135]:
%pip install requests pandas python-dotenv
%pip install jupyter


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Next let's add our helper functions

In [136]:
# HELPER FUNCTIONS
import json

def prettyPrintJSON(data) -> None:
    print(json.dumps(data, indent=2, ensure_ascii=False))

def replace_space_with_percent20(value: str) -> str:
    return value.replace(" ", "%20")

I can't do it gotta use RapidAPI SkyScrapper API

In [ ]:
# SKYSCRAPER API FUNCTIONS

import os
from typing import TypedDict

import requests
from dotenv import load_dotenv

# Load environment variables from .env in the project root
load_dotenv()

rapidapi_key = os.getenv("RAPIDAPI_KEY")
rapidapi_host = os.getenv("RAPIDAPI_HOST", "sky-scrapper.p.rapidapi.com")
headers={
    "X-RapidAPI-Key": rapidapi_key,
    "X-RapidAPI-Host": rapidapi_host,
}

if not rapidapi_key:
    raise ValueError("RAPIDAPI_KEY is missing. Add it to your .env file.")


def SearchHotels(name_of_hotel: str):
    OutputDict = TypedDict(
        "OutputDict",
        {
            "hierarchy": str,
            "location": dict[float, float],
            "score": int,
            "entityName": str,
            "entityId": str,
            "entityType": str,
            "suggestItem": str,
            "class": str,
            "pois": object,
        },
    )

    query = replace_space_with_percent20(name_of_hotel)
    url = f"https://sky-scrapper.p.rapidapi.com/api/v1/hotels/searchDestinationOrHotel?query={query}"
    try:
        response = requests.get(
            url,
            headers=headers,
        )
        if response.status_code == 200:
            response_json = response.json()
            if response_json.get("data") and len(response_json.get("data")) > 0:
                data = response_json.get("data")
                filtered_hotel_array = [
                    item
                    for item in data
                    if item.get("class") == "Hotel" and "Philippines" in item.get("hierarchy", "")
                ]
                if len(filtered_hotel_array) > 0:
                    OutputDictArray: list[OutputDict] = []
                    for hotel in filtered_hotel_array:
                        result: OutputDict = OutputDict()
                        result["class"] = hotel.get("class", "")
                        result["hierarchy"] = hotel.get("hierarchy", "")
                        result["location"] = str(hotel.get("location")).split(",")
                        result["score"] = hotel.get("score", 0)
                        result["entityName"] = hotel.get("entityName", "")
                        result["entityId"] = hotel.get("entityId", "")
                        result["entityType"] = hotel.get("entityType", "")
                        result["suggestItem"] = hotel.get("suggestItem", "")
                        OutputDictArray.append(result)
                    return OutputDictArray
                else:
                    print("No hotels found in the Philippines.")
                    return None
                    
                
        print(f"Error: {response.status_code} - {response.text}")
        return None
    except Exception as e:
        print(f"[SearchHotel] major error: {e}")
        return None



def SearchHotelRates(entityId: str, checkin_date: str, checkout_date: str):
    OutputDict = TypedDict(
        "OutputDict",
        {
            "partnerName": str, # ex: "Booking.com"
            "partnerLogo": str, # ex: "https://www.skyscanner.com.ph/images/websites/220x80/h_bc.png"
            "partnerId": str, # ex: "h_bc"
            "roomType": str, # ex: "Room"
            "roomPolicies": str, # ex: "Meals not included, Non-Refundable" 
            "deeplink": str, # ex: "www.skyscanner.com.ph/hotel_deeplink/4.0/PH/en-US/PHP/h_bc/223183086/2026-04-09/2026-04-10/hotel/hotel/hotels?legacy_provider_id=13&rooms=1&closed_user_group=apps&hb_data=UftNsivRi8__GVuvjbb0svQIh1c90lOveRm3edoT-VXKrLwYErooShejAZfCNZRwGk8bzoXmXPNLe1EPPgUJv1K-NvLX6w6gvGe68WsUb7ebKa2e5defTLgwGDdFhmweOjUZAzz0twk2tV6KZE6hwbPy4bpELfqFwQifmEO92omHAJDO_yLmodgFuTenSbjG5ZPsLnBGRHxehwT6BGnWPr84YO8eTOaeCe_iJ5wKYs6nx2sv2U4jQIEZkNQZLlvPiG2sRTzgWdPhtvCPQb3qLm4DO0slAbVI4-gSgZtlVjx6eMXI-IgpQnafQy06LqI7iFhaE-6zPgDHkCmc1rtNAH1KmwcIFRK7TurGpsHQKbDE6J8_MMgJMShoyAlcralx3BhWwxuQwWcgajmiwZGC8ZSTyEYPp2GzYaMBsQ6OnAh2u68m&q_datetime_utc=2026-03-10T11%3A02%3A10&ticket_price=1486&quote_id=-5316833542518255697&utid_state=1&channel=iphone&price_without_closed_user_group=1807&tm_place_name=Cebu&deeplink_data=eyJmaWVsZHMiOnsic2lnbmF0dXJlIjoiN2M1NDE1NjQ4MmQ2NzliNWVmYWIxNzI5YWExMDViNGUiLCJibG9ja19pZCI6IjEzOTQ2Mjk4MDFfNDExMzcwODU1XzFfMF8wIn19&utid=8be9828c-4358-0139-5815-c42621680778&tm_stars=0&client_id=skyscanner_app&price_policy=2&pre_redirect_id=c8a4e589-a4a8-413c-e220-3f23d76a0862&tm_city_code=CEBP&max_price=1651&guests=1&request_id=c8a4e589-a4a8-413c-e220-3f23d76a0862&tm_country_code=PH"
            "rawPrice": float, # ex: 1486
            "rawPriceGbp": int, # ex: 19
            "rateBriefFeatures": [], # ex: ["Meals not included", "Non-Refundable", "Room"]
            "isOfficial": bool, # false
            "isShowHotelName": bool, # false
            "score": int, # 2
            "cugRate": {
                "discount": str,
                "cugWithoutLabel": str,
                "FSSInfo": object,
                "saveAmount": str,
                "rawSaveAmount": float,
                "type": str, # ex: "apps"
                "discountPercentage": float # type: ignore # 18
            }
        }
    )

    url = f"https://sky-scrapper.p.rapidapi.com/api/v1/hotels/getHotelPrices?hotelId={entityId}&entityId={entityId}&checkin={checkin_date}&checkout={checkout_date}&adults=1&rooms=1&currency=PHP&market=en-US&countryCode=PH"
    try:
        response = requests.get(
            url,
            headers=headers,
        )
        if response.status_code == 200:
            response_json = response.json()

            prettyPrintJSON(response_json)
            
            if (response_json.get("data")):
                data = response_json["data"]
                if (data.get("metaInfo")):
                    metaInfo = data["metaInfo"]
                    if (metaInfo.get("rates")):
                        rates = metaInfo["rates"]
                        OutputDictArray: list[OutputDict] = []
                        for rate in rates:
                            result: OutputDict = rate
                            OutputDictArray.append(result)
                        return OutputDictArray
        print(f"Error: {response.status_code} - {response.text}")
        return None
    except Exception as e:
        print(f"[SearchHotel] major error: {e}")
        return None


In [ ]:
import datetime

import pandas as pd
from enum import Enum
from typing import TypedDict


# Read a specific sheet from an .xlsm file
inputDF = pd.read_excel( "input-data/main.xlsm" )
inputDF = inputDF[inputDF[ "source" ].str.contains( "skyscanner" , na=False)]  # contains the word skyscanner

# TYPED DICTS
class Address(TypedDict):
    city: str
    second_level_nation_administrative_division: str
    first_level_nation_administrative_division: str
    country: str

# ENUMS
class PhilippinesRegion(str, Enum):
    NCR = "NCR"
    CAR = "CAR"
    REGION_I = "Region I"
    REGION_II = "Region II"
    REGION_III = "Region III"
    REGION_IV_A = "Region IV-A"
    REGION_IV_B = "Region IV-B"
    REGION_V = "Region V"
    REGION_VI = "Region VI"
    REGION_VII = "Region VII"
    REGION_VIII = "Region VIII"
    REGION_IX = "Region IX"
    REGION_X = "Region X"
    REGION_XI = "Region XI"
    REGION_XII = "Region XII"
    REGION_XIII = "Region XIII"
    BARMM = "BARMM"

class AreaType(str, Enum):
    URBAN = "Urban"
    SUBURBAN = "Suburban"
    RURAL = "Rural"

class Hotel(TypedDict):
    name: str
    address: Address
    region: PhilippinesRegion
    area_type: AreaType

for index, row in inputDF.iterrows():
    print(row["name_of_hotel"])
    responseHotels = SearchHotels(row["name_of_hotel"])
    if (responseHotels != None):
        print(responseHotels)
        print("len:", len(responseHotels))
        for hotel in responseHotels:
            entityId = hotel["entityId"]

            random_checkin_date = (datetime.datetime.now() + datetime.timedelta(days=30)).strftime("%Y-%m-%d")
            random_checkout_date = (datetime.datetime.now() + datetime.timedelta(days=31)).strftime("%Y-%m-%d")

            responseHotelRates = SearchHotelRates(entityId, random_checkin_date, random_checkout_date)

            if (responseHotelRates != None):
                for rate in responseHotelRates:
                    rate[""]
            
            print("--------------")

        


    break

CF Haven - Near IT Park, Ayala, SM
[{'class': 'Hotel', 'hierarchy': 'Cebu|Consolacion|Philippines', 'location': ['10.32589', ' 123.91095'], 'score': 0, 'entityName': 'CF Haven - Near IT Park, Ayala, SM', 'entityId': '223183086', 'entityType': 'hotel', 'suggestItem': '{strong}CF{/strong} {strong}Haven{/strong} - {strong}Near{/strong} {strong}IT{/strong} {strong}Park{/strong}, {strong}Ayala{/strong}, {strong}SM{/strong}, Cebu, Consolacion, Philippines'}]
len: 1
{
  "status": true,
  "timestamp": 1773141400352,
  "data": {
    "metaInfo": {
      "ratesCta": "Go to site",
      "rates": [
        {
          "partnerName": "Booking.com",
          "partnerLogo": "https://www.skyscanner.com.ph/images/websites/220x80/h_bc.png",
          "partnerId": "h_bc",
          "roomType": "Room",
          "roomPolicies": "Meals not included, Non-Refundable",
          "deeplink": "www.skyscanner.com.ph/hotel_deeplink/4.0/PH/en-US/PHP/h_bc/223183086/2026-04-09/2026-04-10/hotel/hotel/hotels?legacy_pr

Below we'll be doing ThreadPoolExecutor on this - it's safe to output every process I think

After processing finally output the values

These data values from the google (2) were taken from February 18 - February 24. 

These values are checkin and checkout days of 2 days and 1 night. So that's today and tomorrow checkin and checkout.

I'll be using Ollama to finish all the area-types which are:  Urban ,  Suburban ,  Rural 

--------------------

Let's also create a table with empty values for the prediction model to predict on. 


Step 1: For every row with empty rooms fill it up with a random room type that is available from the non-empty rows

Step 2: Output the table with filled rooms but with empty room prices. We will predict their room price from there.